In [1]:
 !pip install -U bitsandbytes>=0.46.1 accelerate

In [ ]:
!pip install trl

In [3]:
import json
import torch
import trl
import numpy as np
from pathlib import Path
from collections import Counter
from sklearn.model_selection import train_test_split

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    EarlyStoppingCallback,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

In [4]:
MODEL_ID   = "zou-lab/BioMed-R1-8B"
DATA_PATH  = "/content/drive/MyDrive/BioMed AI CW/FineTune Dataset/ori_pqal.json"
OUTPUT_DIR = "/content/drive/MyDrive/BioMed AI CW/biomed_r1_finetuned"
MAX_SEQ_LEN = 512       # keep short for T4 VRAM budget
NUM_EPOCHS  = 3
BATCH_SIZE  = 1         # must be 1 on T4 for 8B model
GRAD_ACCUM  = 16        # effective batch size = 1 × 16 = 16
LR          = 2e-4
WARMUP_RATIO = 0.05
LORA_R      = 16
LORA_ALPHA  = 32
LORA_DROPOUT = 0.05
SEED        = 42

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

In [5]:
print("Loading ori_pqal.json...")
with open(DATA_PATH, 'r') as f:
    raw_data = json.load(f)

print(f"Total samples: {len(raw_data)}")
labels_all = [v['final_decision'].lower().strip() for v in raw_data.values()]
print(f"Label distribution: {Counter(labels_all)}")

Loading ori_pqal.json...
Total samples: 1000
Label distribution: Counter({'yes': 552, 'no': 338, 'maybe': 110})


In [6]:
def build_context(labels: list, contexts: list) -> str:
    """Zip section headings with paragraph text into a structured abstract."""
    parts = []
    for label, text in zip(labels, contexts):
        parts.append(f"{label.strip().title()}: {text.strip()}")
    return "\n".join(parts)

In [7]:
def format_training_sample(entry: dict) -> str:
    """
    Format a single ori_pqal entry into a complete prompt+answer string.
    During fine-tuning the model sees the full string including the answer,
    so it learns to conclude with 'Final Answer: <label>'.
    """
    question = entry.get('QUESTION', entry.get('question', ''))
    raw_labels   = entry.get('LABELS',   [])
    raw_contexts = entry.get('CONTEXTS', [])

    if isinstance(raw_contexts, list) and isinstance(raw_labels, list):
        context = build_context(raw_labels, raw_contexts)
    elif isinstance(raw_contexts, list):
        context = " ".join(raw_contexts)
    else:
        context = str(raw_contexts)

    gold = entry.get('final_decision', '')
    if isinstance(gold, list):
        gold = gold[0]
    gold = gold.lower().strip()

    # The full training string: prompt + correct answer
    # The model learns that after this reasoning instruction, it should output
    # a brief analysis followed by the Final Answer line.
    prompt = (
        f"The following is a structured biomedical abstract:\n\n"
        f"{context}\n\n"
        f"Based ONLY on the evidence in the abstract above, answer this question:\n"
        f"{question}\n\n"
        f"Instructions:\n"
        f"- Think through the evidence step by step.\n"
        f"- Your final answer MUST be exactly one of: yes, no, or maybe.\n"
        f"- End your response with exactly one of these three lines:\n"
        f"  Final Answer: yes\n"
        f"  Final Answer: no\n"
        f"  Final Answer: maybe\n\n"
        f"Final Answer: {gold}"   # ← supervision signal
    )
    return prompt

In [8]:
print("\nBuilding dataset...")
samples = []
for key, entry in raw_data.items():
    gold = entry.get('final_decision', '')
    if isinstance(gold, list): gold = gold[0]
    gold = gold.lower().strip()

    text = format_training_sample(entry)
    samples.append({"text": text, "label": gold, "pmid": key})

# Stratified split — preserves yes/no/maybe ratios in both train and val
train_samples, val_samples = train_test_split(
    samples,
    test_size=0.1,          # 900 train, 100 val
    random_state=SEED,
    stratify=[s["label"] for s in samples]
)

print(f"Train: {len(train_samples)} samples")
print(f"  Distribution: {Counter(s['label'] for s in train_samples)}")
print(f"Val:   {len(val_samples)} samples")
print(f"  Distribution: {Counter(s['label'] for s in val_samples)}")

train_dataset = Dataset.from_list([{"text": s["text"]} for s in train_samples])
val_dataset   = Dataset.from_list([{"text": s["text"]} for s in val_samples])


Building dataset...
Train: 900 samples
  Distribution: Counter({'yes': 497, 'no': 304, 'maybe': 99})
Val:   100 samples
  Distribution: Counter({'yes': 55, 'no': 34, 'maybe': 11})


In [9]:
print("\nLoading model with 4-bit quantisation...")
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,   # float16 for T4 (no bfloat16 support)
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
# SFTTrainer needs a pad token — use eos if none set
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    device_map="auto",
    dtype=torch.float16,
    trust_remote_code=True,
)
for param in model.parameters():
    if param.dtype == torch.bfloat16:
        param.data = param.data.to(torch.float16)

model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
)


Loading model with 4-bit quantisation...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [10]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",   # attention
        "gate_proj", "up_proj", "down_proj",        # MLP
    ],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 41,943,040 || all params: 8,072,204,288 || trainable%: 0.5196


In [11]:
# ── 8. SFTConfig (trl 1.1.0) ─────────────────────────────────────────────────
sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,

    # Batch & memory
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},

    # Optimiser
    optim="paged_adamw_8bit",
    learning_rate=LR,
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    warmup_steps=WARMUP_RATIO,
    max_grad_norm=0.3,

    # Precision — float16 for T4
    fp16=False,
    bf16=False,

    # Evaluation & checkpointing
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # SFT-specific
    dataset_text_field="text",
    packing=False,
    completion_only_loss=False,

    # Logging
    logging_dir=f"{OUTPUT_DIR}/logs",
    logging_steps=10,
    report_to="none",

    seed=SEED,
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [12]:
# ── 9. Trainer ────────────────────────────────────────────────────────────────
# FIX 1: use processing_class= not tokenizer= (renamed in trl 1.1.0)
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,        # FIX 1: was tokenizer=tokenizer
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

Adding EOS to train dataset:   0%|          | 0/900 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/900 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

In [13]:
steps_per_epoch = len(train_dataset) // (BATCH_SIZE * GRAD_ACCUM)
total_steps     = steps_per_epoch * NUM_EPOCHS

print(f"\nStarting fine-tuning...")
print(f"  Effective batch size : {BATCH_SIZE * GRAD_ACCUM}")
print(f"  Steps per epoch      : {steps_per_epoch}")
print(f"  Total steps          : {total_steps}")
print(f"  Eval every           : 50 steps")
print(f"  Expected time on T4  : ~2.5–3.5 hours\n")

trainer.train()


Starting fine-tuning...
  Effective batch size : 16
  Steps per epoch      : 56
  Total steps          : 168
  Eval every           : 50 steps
  Expected time on T4  : ~2.5–3.5 hours



Step,Training Loss,Validation Loss
50,1.304071,1.314403


Step,Training Loss,Validation Loss
50,1.304071,1.314403
100,1.208442,1.334029
150,1.057597,1.378410


TrainOutput(global_step=171, training_loss=1.2197911439583315, metrics={'train_runtime': 7526.5378, 'train_samples_per_second': 0.359, 'train_steps_per_second': 0.023, 'total_flos': 5.033875476293222e+16, 'train_loss': 1.2197911439583315})

In [14]:
print("\nSaving LoRA adapter...")
adapter_path = f"{OUTPUT_DIR}/final_adapter"
trainer.model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f"Adapter saved to: {adapter_path}")
print("(This is ~80-150MB — the LoRA weights only, not the full 8B model)")


Saving LoRA adapter...
Adapter saved to: /content/drive/MyDrive/BioMed AI CW/biomed_r1_finetuned/final_adapter
(This is ~80-150MB — the LoRA weights only, not the full 8B model)
